Завантаження та нормалізація даних MNIST

In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from keras.datasets import mnist
from keras.models import Sequential, load_model
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, Input
from keras import utils
from keras.utils import to_categorical, load_img, img_to_array
from sklearn.model_selection import KFold

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0

print(f"Розмірність навчальних даних: {X_train.shape}")
print(f"Розмірність тестових даних: {X_test.shape}")


Розмірність навчальних даних: (60000, 28, 28)
Розмірність тестових даних: (10000, 28, 28)


Клас Callback та навчання моделі до 99% точності

In [5]:
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if logs.get('accuracy') is not None and logs.get('accuracy') >= 0.99:
            print("\nReached 99% accuracy so cancelling training!")
            print(f"Досягнуто 99% точності на епосі {epoch + 1}, тому скасовуємо навчання!")
            self.model.stop_training = True

def train_mnist(x_train, y_train):
    callbacks = myCallback()
    
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(512, activation='relu'),
        Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    
    history = model.fit(x_train, y_train, epochs=10, callbacks=[callbacks], verbose=1)
    return history

hist = train_mnist(X_train_norm, y_train)


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9410 - loss: 0.2008
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9758 - loss: 0.0791
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9841 - loss: 0.0515
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9879 - loss: 0.0373
Epoch 5/10
1869/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9915 - loss: 0.0265
Reached 99% accuracy so cancelling training!
Досягнуто 99% точності на епосі 5, тому скасовуємо навчання!
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9915 - loss: 0.0265


Візуалізація перших 4 зображень

In [ ]:
def simple_cnn():
    model = Sequential([
        Conv2D(32, (5, 5), input_shape=(28, 28, 1), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.2),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

model_simple = simple_cnn()
model_simple.summary()
history_simple = model_simple.fit(X_train_cnn, y_train_cat, validation_data=(X_test_cnn, y_test_cat), epochs=10, batch_size=200, verbose=2)

scores_simple = model_simple.evaluate(X_test_cnn, y_test_cat, verbose=0)
print(f"Simple CNN Error: {100 - scores_simple[1] * 100:.2f}%")
print(f"Simple CNN Accuracy: {scores_simple[1] * 100:.2f}%")


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 592,074 (2.26 MB)

 Trainable params: 592,074 (2.26 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
300/300 - 7s - 22ms/step - accuracy: 0.9302 - loss: 0.2440 - val_accuracy: 0.9760 - val_loss: 0.0792
Epoch 2/10


# ============================================================
# ЗАВДАННЯ 5.3: Згорткова нейромережа CNN з callback на 99.5%
# ============================================================
# Підготовка 4D тензорів [samples, 28, 28, 1] для згорткових шарів
X_train_cnn = X_train_norm.reshape(-1, 28, 28, 1).astype('float32')
X_test_cnn = X_test_norm.reshape(-1, 28, 28, 1).astype('float32')

# Зворотний виклик для порогу точності 99.5%
class myCallback995(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if logs.get('accuracy') is not None and logs.get('accuracy') >= 0.995:
            print("\nReached 99.5% accuracy so cancelling training!")
            print(f"Досягнуто 99,5% точності на епосі {epoch + 1}, тому скасовуємо навчання!")
            self.model.stop_training = True

# Модель із Завдання 1 з додаванням 1 шару Conv2D та 1 шару MaxPooling2D
model_cnn_cb = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(10, activation='softmax')
])

model_cnn_cb.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Тренування з перериванням при досягненні 99.5% точності
model_cnn_cb.fit(X_train_cnn, y_train, epochs=10, batch_size=128, callbacks=[myCallback995()], verbose=1)


In [ ]:
def deeper_cnn():
    model = Sequential([
        Conv2D(30, (5, 5), input_shape=(28, 28, 1), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(15, (3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.2),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(50, activation='relu'),
        Dense(10, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

model_deeper = deeper_cnn()
model_deeper.summary()
history_deeper = model_deeper.fit(X_train_cnn, y_train_cat, validation_data=(X_test_cnn, y_test_cat), epochs=10, batch_size=200, verbose=2)

scores_deeper = model_deeper.evaluate(X_test_cnn, y_test_cat, verbose=0)
print(f"Deeper CNN Error: {100 - scores_deeper[1] * 100:.2f}%")
print(f"Deeper CNN Accuracy: {scores_deeper[1] * 100:.2f}%")


Model: "sequential_4"
+--------------------------------------------------------------------------+
| Layer (type)                    | Output Shape           |       Param # |
|---------------------------------+------------------------+---------------|
| conv2d_2 (Conv2D)               | (None, 24, 24, 30)     |           780 |
|---------------------------------+------------------------+---------------|
| max_pooling2d_2 (MaxPooling2D)  | (None, 12, 12, 30)     |             0 |
|---------------------------------+------------------------+---------------|
| conv2d_3 (Conv2D)               | (None, 10, 10, 15)     |         4,065 |
|---------------------------------+------------------------+---------------|
| max_pooling2d_3 (MaxPooling2D)  | (None, 5, 5, 15)       |             0 |
|---------------------------------+------------------------+---------------|
| dropout_1 (Dropout)             | (None, 5, 5, 15)       |             0 |
|---------------------------------+-------------------